In [ ]:
# Global Oil and Gas Exposure Study
# 00: PopExposure Set Up

# This file sets up environment to apply Popexposure to finds the number of people living near buffered oil and gas wells. 
# Author: Lara Schwarz
# Last updated: October 2025
# Reviewed by Nina Flores: September 2025

# Steps included in this script:

## Set up step 1: Importing libraries
## Set up step 2: Defining the environments
## Set up step 3: Loading global social data 
## Set up step 4: Loading global land area data

In [ ]:
## Set up step 1: Importing libraries

import geopandas as gpd
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from thefuzz import process
import pandas as pd
import pathlib
import pandas as pd
import sys
import os
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import geopandas as gpd
from rasterio.plot import show
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import fiona
import geopandas as gpd
from shapely.geometry import Polygon
import ee
import geemap
import matplotlib.cm as cm
import matplotlib.colors as colors
from shapely.geometry import box
import math
from shapely.geometry import Point
import glob
import seaborn as sns
import openpyxl
import subprocess
from pathlib import Path
from popexposure import PopEstimator

In [ ]:
## Set up step 2: defining the environments
# Define base paths
repo = pathlib.Path.cwd().parent.parent
share_path = pathlib.Path("/Users/larasch/Library/CloudStorage/OneDrive-SharedLibraries-UW/casey_cohort - Documents/data")
gee_path= pathlib.Path("Users/larasch/Library/CloudStorage/GoogleDrive-laranschwarz@gmail.com/My Drive/EarthEngineExports")

# Define code path and add to sys.path
code_path = repo / "code"

# Define data paths
# location of the oil and gas data
oil_path = share_path / "environmental/oil_and_gas/raw_data/ogim.parquet"

# location of the adminstrative boundary shapefile
countries_path = share_path / "geo_boundaries/processed_data/country_admin/country_geom_filtered.parquet"

# location of the population data
ghs_pop_path = share_path / "social_including_census/raw_data/GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0/GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0.tif"

# Import local modules
from popexposure import *

In [ ]:
## Set up step 3: Loading global social data 
## Importing global country shapefiles 

# Define the paths
csv_path = repo / "data/IHME_LSAE_admin2_LDIpc_2019_with_loc_id(in)_v2.csv"  # Adjust the path accordingly

# Load the CSV data
# Try reading the CSV with a different encoding (ISO-8859-1 or utf-16)
try:
    csv_data = pd.read_csv(csv_path, encoding='ISO-8859-1')  # or try encoding='utf-16'
    print("CSV loaded successfully with encoding ISO-8859-1.")
except UnicodeDecodeError:
    print("Error: Unable to decode CSV with the provided encoding.")

# Check the first few rows of the CSV
print(csv_data.head())

# Load the shapefile (GeoDataFrame)
shapefile_path = repo/ "data/OneDrive_1_12-11-2024/lbd_standard_admin_2.shp"

global_shapefiles = gpd.read_file(shapefile_path)

# merge the data
merged_shapefile_social = global_shapefiles.merge(csv_data, how='left', left_on='loc_id', right_on='loc_id')

# Now you can plot the data on the map
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
merged_shapefile_social.plot(column='LDIpc_mean', ax=ax, legend=True,
                legend_kwds={'label': "LDIpc by Country",
                             'orientation': "horizontal"})
plt.title("LDIpc Data Map")
plt.show()

global_shapefiles.plot()


In [ ]:
## Set up step 4: Loading global land area data (land mask) for use as raster in function

# Define Paths
tif_files = sorted(glob.glob(str(gee_path / "MOD44W_2015_global-*.tif")))
merged_file = gee_path / "MOD44W_2015_water_mask_merged.tif"
clipped_file = gee_path / "MOD44W_2015_water_mask_clipped_anyoverlap.tif"
clip_vector = share_path / "geo_boundaries/processed_data/countries_ogd/all_countries_3km_buffer.geojson"

ee.Initialize()

mod44w = (ee.ImageCollection('MODIS/006/MOD44W')
          .filterDate('2015-01-01', '2015-12-31')
          .first()
          .select('water_mask'))

region = ee.Geometry.Rectangle([-180, -90, 180, 90], geodesic=False)

task = ee.batch.Export.image.toDrive(
    image=mod44w,
    description='MOD44W_WaterMask_2015_Global',
    folder='EarthEngineExports',
    fileNamePrefix='MOD44W_2015_global',
    region=region,
    scale=250,
    maxPixels=1e13
)
task.start()

#  Merge raster tiles
print("Merging raster tiles...")
subprocess.run(["gdal_merge.py", "-o", str(merged_file)] + tif_files, check=True)

#  Clip + mask, keeping ANY intersecting grid cells
print("Clipping and masking to vector extent (include any overlap)...")
subprocess.run([
    "gdalwarp",
    "-cutline", clip_vector,
    "-crop_to_cutline",               # ensures bounding box includes full shape
    "-dstnodata", "255",              # MOD44W NoData value
    "-overwrite",
    "-of", "GTiff",
    str(merged_file),
    str(clipped_file)#], check=True)

# Invert raster so land=1, water=0 
inverted_file = gee_path / "MOD44W_2015_land_mask_clipped_anyoverlap.tif"

subprocess.run([
    "gdal_calc.py",
    "-A", str(clipped_file),
    "--outfile", str(inverted_file),
    "--calc", "1-A",
    "--NoDataValue=255",
    "--overwrite"
], check=True)

print(f"Inverted raster (land=1, water=0) saved to:\n{inverted_file}")
print(f"Raster clipped with any overlapping cells saved to:\n{clipped_file}")